In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from scipy.interpolate import UnivariateSpline
import astropy.io.fits as fits
import astropy.units as u
from astropy.coordinates import SkyCoord
from sunpy.coordinates import Helioprojective
import sunpy
import sunpy.map
from skimage.transform import resize
from sunkit_instruments import suvi
from aiapy.calibrate import register, update_pointing
from scipy import stats
import sunpy.sun.constants as const
from copy import deepcopy
from scipy.ndimage import gaussian_filter
from PIL import Image
import matplotlib.colors as colors
from astropy.visualization import ImageNormalize, SqrtStretch, LogStretch, PercentileInterval

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

data_dir = '/home/mnedal/data'

In [2]:
def split_datetime(start=None, end=None):
    
    START_DATE, START_TIME = start.split('T')
    END_DATE, END_TIME = end.split('T')

    START_YEAR, START_MONTH, START_DAY = START_DATE.split('-')
    END_YEAR, END_MONTH, END_DAY = END_DATE.split('-')

    START_HOUR, START_MINUTE, START_SECOND = START_TIME.split(':')
    END_HOUR, END_MINUTE, END_SECOND = END_TIME.split(':')

    datetime_dict = {
        'start_year': START_YEAR,
        'start_month': START_MONTH,
        'start_day': START_DAY,
        'start_hour': START_HOUR,
        'start_minute': START_MINUTE,
        'start_second': START_SECOND,
        
        'end_year': END_YEAR,
        'end_month': END_MONTH,
        'end_day': END_DAY,
        'end_hour': END_HOUR,
        'end_minute': END_MINUTE,
        'end_second': END_SECOND
    }
    return datetime_dict




def load_aia_lv1(start=None, end=None, channel=193):
    dt_dict = split_datetime(start=start, end=end)
    data_path = f'{data_dir}/AIA/{channel}A'
    data = sorted(glob.glob(f'{data_path}/aia_lev1_{channel}a_*.fits'))
    
    start_filename = f"aia_lev1_{channel}a_{dt_dict['start_year']}_{dt_dict['start_month']}_{dt_dict['start_day']}t{dt_dict['start_hour']}_{dt_dict['start_minute']}"
    end_filename   = f"aia_lev1_{channel}a_{dt_dict['end_year']}_{dt_dict['end_month']}_{dt_dict['end_day']}t{dt_dict['end_hour']}_{dt_dict['end_minute']}"
    
    first_file_to_find = glob.glob(f'{data_path}/{start_filename}*.fits')
    last_file_to_find  = glob.glob(f'{data_path}/{end_filename}*.fits')
    
    idx1 = data.index(first_file_to_find[0])
    idx2 = data.index(last_file_to_find[0])
    
    chosen_files = data[idx1:idx2]
    
    map_objects = []
    for i, file in enumerate(chosen_files):
        # load the file as a sunpy map
        m = sunpy.map.Map(file)
        map_objects.append(m)
        print(f'AIA {channel}A image {i} is loaded')
    return map_objects




def load_aia(start=None, end=None, channel=193):
    dt_dict = split_datetime(start=start, end=end)
    data_path = f'{data_dir}/AIA/{channel}A/lv15'
    data = sorted(glob.glob(f'{data_path}/aia_{channel}a_*.fits'))
    
    start_filename = f"aia_{channel}a_{dt_dict['start_year']}_{dt_dict['start_month']}_{dt_dict['start_day']}T{dt_dict['start_hour']}_{dt_dict['start_minute']}"
    end_filename   = f"aia_{channel}a_{dt_dict['end_year']}_{dt_dict['end_month']}_{dt_dict['end_day']}T{dt_dict['end_hour']}_{dt_dict['end_minute']}"
    
    first_file_to_find = glob.glob(f'{data_path}/{start_filename}*.fits')
    last_file_to_find  = glob.glob(f'{data_path}/{end_filename}*.fits')
    
    idx1 = data.index(first_file_to_find[0])
    idx2 = data.index(last_file_to_find[0])
    
    chosen_files = data[idx1:idx2]
    
    map_objects = []
    for i, file in enumerate(chosen_files):
        # load the file as a sunpy map
        m = sunpy.map.Map(file)
        map_objects.append(m)
        print(f'AIA {channel}A image {i} is loaded')
    return map_objects



# make running-ratio images
def apply_runratio(maps):
    """
    Apply running-ratio image technique on EUV images.
    See: https://iopscience.iop.org/article/10.1088/0004-637X/750/2/134/pdf
    Inputs: list of EUV sunpy maps.
    Output: sequence of run-ratio sunpy maps.
    """
    runratio = [m / prev_m.quantity for m, prev_m in zip(maps[1:], maps[:-1])]
    m_seq_runratio = sunpy.map.Map(runratio, sequence=True)
    
    # for m in m_seq_runratio:
    #     m.data[np.isnan(m.data)] = 1
    #     m.plot_settings['norm'] = colors.Normalize(vmin=0, vmax=2)
    #     m.plot_settings['cmap'] = 'seismic'
    
    return m_seq_runratio



def enhance_contrast(image, vmin, vmax):
    """
    Enhance contrast by clipping and normalization.
    """
    image_clipped = np.clip(image, vmin, vmax)
    image_normalized = (image_clipped - vmin) / (vmax - vmin)
    return image_normalized



def calculate_percentiles(image, lower_percentile=3, upper_percentile=97):
    """
    Calculate vmin and vmax based on the 1st and 99th percentiles.
    """
    vmin = np.percentile(image, lower_percentile)
    vmax = np.percentile(image, upper_percentile)
    return vmin, vmax



def round_obstime(time=None):
    """
    Round the observation time to put it in the image title.
    Input : str, time (HH:MM:SS.sss)
    Output: str, datetime (YYYY-mm-DD HH:MM:SS)
    """
    from datetime import datetime, timedelta

    original_time_str = time

    # Convert the original time string to a datetime object
    original_time = datetime.strptime(original_time_str, '%H:%M:%S.%f')

    # Round the time to the nearest second
    rounded_time = original_time + timedelta(seconds=round(original_time.microsecond / 1e6))

    # Format the rounded time as 'HH:MM:SS'
    rounded_time_str = rounded_time.strftime('%H:%M:%S')
    
    return rounded_time_str



def have_same_shape(*arrays):
    """
    Check if the given arrays have the same shape.
    """
    first_shape = arrays[0].data.shape
    return all(arr.data.shape == first_shape for arr in arrays)




def plot_line(angle_deg=None, length=None, map_obj=None):
    """
    Plot a straight line at an angle in degrees from the solar West.
    """
    angle_rad = np.deg2rad(angle_deg)

    # Define the length of the line (in arcseconds)
    line_length = length * u.arcsec

    # Define the center point of the line (e.g., the center of the map)
    center = map_obj.center

    # Calculate the start and end points of the line
    start_point = SkyCoord(center.Tx, center.Ty, frame=map_obj.coordinate_frame)

    end_point = SkyCoord(center.Tx + line_length * np.cos(angle_rad),
                        center.Ty + line_length * np.sin(angle_rad),
                        frame=map_obj.coordinate_frame)
    
    line = SkyCoord([start_point, end_point])
    return line



def generate_centered_list(center, difference, num_elements):
    """
    Generate a list of numbers centered around a given number with a specified difference
    between consecutive numbers.

    Parameters:
    center (int): The central number around which the list is generated.
    difference (int): The difference between consecutive numbers in the list.
    num_elements (int): The number of slits before and after the central slit.

    Returns:
    list: A list of numbers centered around the specified central number.
    """
    return [center + difference * i for i in range(-num_elements, num_elements + 1)]

In [4]:
# aia_131_map_objects = load_aia(start='2024-05-14T17:13:00', end='2024-05-14T18:33:00', channel=131)
aia_171_map_objects = load_aia(start='2024-05-14T17:13:00', end='2024-05-14T17:45:00', channel=171)
aia_193_map_objects = load_aia(start='2024-05-14T17:13:00', end='2024-05-14T17:45:00', channel=193)
aia_211_map_objects = load_aia(start='2024-05-14T17:13:00', end='2024-05-14T17:45:00', channel=211)

IndexError: list index out of range

In [ ]:
# print(len(aia_131_map_objects),
len(aia_171_map_objects), len(aia_193_map_objects), len(aia_211_map_objects)

In [ ]:
idx = 16

fig = plt.figure(figsize=[15,5])
fig.suptitle('Before manual normalization', y=0.95)

# ax = fig.add_subplot(141, projection=aia_131_map_objects[idx])
# img = aia_131_map_objects[idx].plot(axes=ax)
# plt.colorbar(img, shrink=0.5, pad=0.02)
# ax.grid(False)

ax = fig.add_subplot(131, projection=aia_171_map_objects[idx])
img = aia_171_map_objects[idx].plot(axes=ax)
plt.colorbar(img, shrink=0.5, pad=0.02)
ax.grid(False)

ax = fig.add_subplot(132, projection=aia_193_map_objects[idx])
img = aia_193_map_objects[idx].plot(axes=ax)
plt.colorbar(img, shrink=0.5, pad=0.02)
ax.grid(False)

ax = fig.add_subplot(133, projection=aia_211_map_objects[idx])
img = aia_211_map_objects[idx].plot(axes=ax)
plt.colorbar(img, shrink=0.5, pad=0.02)
ax.grid(False)

fig.tight_layout()
plt.show()

In [ ]:
# for m in aia_131_map_objects:
#     m.plot_settings['norm'] = ImageNormalize(vmin=0, vmax=2e3, stretch=LogStretch())

for m in aia_171_map_objects:
    m.plot_settings['norm'] = ImageNormalize(vmin=0, vmax=4e3, stretch=LogStretch())

for m in aia_193_map_objects:
    m.plot_settings['norm'] = ImageNormalize(vmin=0, vmax=4e3, stretch=LogStretch())

for m in aia_211_map_objects:
    m.plot_settings['norm'] = ImageNormalize(vmin=0, vmax=4e3, stretch=LogStretch())

In [ ]:
idx = 16

fig = plt.figure(figsize=[15,5])
fig.suptitle('After manual normalization', y=0.95)

# ax = fig.add_subplot(141, projection=aia_131_map_objects[idx])
# img = aia_131_map_objects[idx].plot(axes=ax)
# plt.colorbar(img, shrink=0.5, pad=0.02)
# ax.grid(False)

ax = fig.add_subplot(131, projection=aia_171_map_objects[idx])
img = aia_171_map_objects[idx].plot(axes=ax)
plt.colorbar(img, shrink=0.5, pad=0.02)
ax.grid(False)

ax = fig.add_subplot(132, projection=aia_193_map_objects[idx])
img = aia_193_map_objects[idx].plot(axes=ax)
plt.colorbar(img, shrink=0.5, pad=0.02)
ax.grid(False)

ax = fig.add_subplot(133, projection=aia_211_map_objects[idx])
img = aia_211_map_objects[idx].plot(axes=ax)
plt.colorbar(img, shrink=0.5, pad=0.02)
ax.grid(False)

fig.tight_layout()
plt.show()

### Discard the images with low time exposure

In [ ]:
exposure_threshold = 0.5 # in seconds

bad_171_images = []
for i, m in enumerate(aia_171_map_objects):
    if m.meta['exptime'] < exposure_threshold:
        # print(f"image {i} is bad at {m.meta['t_obs']}")
        bad_171_images.append(i)

bad_193_images = []
for i, m in enumerate(aia_193_map_objects):
    if m.meta['exptime'] < exposure_threshold:
        # print(f"image {i} is bad at {m.meta['t_obs']}")
        bad_193_images.append(i)

bad_211_images = []
for i, m in enumerate(aia_211_map_objects):
    if m.meta['exptime'] < exposure_threshold:
        # print(f"image {i} is bad at {m.meta['t_obs']}")
        bad_211_images.append(i)

unique_indices = set(bad_171_images + bad_193_images + bad_211_images)
# unique_indices = set(bad_193_images + bad_211_images)
indices_to_remove = list(unique_indices)
print(f'Images to discard: {len(indices_to_remove)}')

# Make a new list excluding the elements at the specified indices
clean_runratio_171A = [item for index, item in enumerate(aia_171_map_objects) if index not in indices_to_remove]
clean_runratio_193A = [item for index, item in enumerate(aia_193_map_objects) if index not in indices_to_remove]
clean_runratio_211A = [item for index, item in enumerate(aia_211_map_objects) if index not in indices_to_remove]

In [ ]:
print(f'171 A --> Original list: {len(aia_171_map_objects)}, Clean list: {len(clean_runratio_171A)}')
print(f'193 A --> Original list: {len(aia_193_map_objects)}, Clean list: {len(clean_runratio_193A)}')
print(f'211 A --> Original list: {len(aia_211_map_objects)}, Clean list: {len(clean_runratio_211A)}')

In [ ]:
# make run-ratio maps
m_seq_runratio_171A = apply_runratio(clean_runratio_171A)
m_seq_runratio_193A = apply_runratio(clean_runratio_193A)
m_seq_runratio_211A = apply_runratio(clean_runratio_211A)

print(len(m_seq_runratio_171A), len(m_seq_runratio_193A), len(m_seq_runratio_211A))

In [ ]:
m_seq_runratio_171A[0].data.shape, m_seq_runratio_193A[0].data.shape, m_seq_runratio_211A[0].data.shape

In [ ]:
### make RGB images from the clean maps

if have_same_shape(m_seq_runratio_171A[0], m_seq_runratio_193A[0], m_seq_runratio_211A[0]):
    print('All arrays have the same shape')
else:
    print('The arrays do not have the same shape')

    rgb_maps = []

    for idx, m in enumerate(m_seq_runratio_171A): # work with all images
        # Determine the target shape (smallest shape among the three images)
        target_shape = (min(m_seq_runratio_171A[idx].data.shape[0], m_seq_runratio_193A[idx].data.shape[0], m_seq_runratio_211A[idx].data.shape[0]),
                        min(m_seq_runratio_171A[idx].data.shape[1], m_seq_runratio_193A[idx].data.shape[1], m_seq_runratio_211A[idx].data.shape[1]))
        
        # Resize each image to the target shape
        data_171A_resized = resize(m_seq_runratio_171A[idx].data, target_shape, preserve_range=True)
        data_193A_resized = resize(m_seq_runratio_193A[idx].data, target_shape, preserve_range=True)
        data_211A_resized = resize(m_seq_runratio_211A[idx].data, target_shape, preserve_range=True)
        
        # Stack the resized images
        rgb_image = np.stack([data_171A_resized, data_193A_resized, data_211A_resized], axis=-1)
        
        # Convert the 3D array to a 2D array by averaging the channels
        array_2d = np.mean(rgb_image, axis=2)
        
        # Make a SunPy map with the resulting array
        final_map = sunpy.map.Map(array_2d, m_seq_runratio_171A[idx].meta)
        rgb_maps.append(final_map)
        print(f'image {idx} is done')

print(len(rgb_maps))

In [ ]:
# Show an example image
m = rgb_maps[7]

m.data[np.isnan(m.data)] = 1
m.plot_settings['cmap'] = 'seismic'
m.plot_settings['norm'] = ImageNormalize(vmin=0, vmax=5, stretch=SqrtStretch())

fig = plt.figure(figsize=[7,7])
ax = fig.add_subplot(111, projection=m)
m.plot(axes=ax)
m.draw_limb()

date_str = str(m.date).split('T')[0]
rounded_time_str = round_obstime(time=str(m.date).split('T')[1])
ax.set_title(f'AIA 171, 193, 211 $\AA$ {date_str} {rounded_time_str}')

# Plot the slits
centered_list = generate_centered_list(161, 4, 3)

for angle in centered_list:
    line = plot_line(angle_deg=angle, length=1450, map_obj=m)
    ax.plot_coord(line, color='black')
    
    # Plot the number at the end of the line
    # Convert SkyCoord to pixel coordinates for plotting text
    line_lon, line_lat = line.Tx, line.Ty
    end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
    
    # Display the number at the end point
    ax.text(end_point_pixel.x.value - 50, end_point_pixel.y.value + 50, f'{angle}$^o$',
            color='black', fontsize=10, ha='center', va='center')

plt.show()

In [ ]:
print(len(rgb_maps))

## Loop over all the images

In [ ]:
# Create a dictionary to hold the lists of lists
intensity_dict = {}
distances_dict = {}
output_obj = {}

# Initialize each key with an empty list
for value in centered_list:
    intensity_dict[f'intensity_{value}deg'] = []
    distances_dict[f'distances_{value}deg'] = []
    
    output_obj[f'intensity_{value}deg'] = []
    output_obj[f'distances_{value}deg'] = []

output_obj['time'] = []
output_obj['map_obj'] = []
output_obj['instrument'] = []

for i, m in enumerate(rgb_maps):
    print(f'Working on map {i} ..')
    
    m.data[np.isnan(m.data)] = 1
    m.plot_settings['cmap'] = 'seismic'
    m.plot_settings['norm'] = ImageNormalize(vmin=0, vmax=5, stretch=SqrtStretch())
    
    fig = plt.figure(figsize=[7,7])
    ax = fig.add_subplot(111, projection=m)
    m.plot(axes=ax)
    
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1450, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 50, end_point_pixel.y.value + 50, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
        
        # obtain the coordinates of the map pixels that intersect that path
        intensity_coords_slit = sunpy.map.pixelate_coord_path(m, line)
        
        # Create mask to identify valid coordinates
        valid_mask = sunpy.map.contains_coordinate(m, intensity_coords_slit)
        
        # Apply the mask to filter valid coordinates
        valid_coords = intensity_coords_slit[valid_mask]
        
        # Pass those coordinates to extract the values for those map pixels
        intensity_slit = sunpy.map.sample_at_coords(m, valid_coords)
        
        # Calculate the angular separation between the first point and every other coordinate we extracted
        angular_separation_slit = valid_coords.separation(valid_coords[0]).to(u.arcsec)
        
        # Append the values to the lists
        intensity_dict[f'intensity_{value}deg'].append(list(intensity_slit.value))
        distances_dict[f'distances_{value}deg'].append(list(angular_separation_slit.value))
    
    output_obj['time'].append(m.date.iso)
    output_obj['map_obj'].append(m)
    output_obj['instrument'].append(m.meta['instrume'])
    
    plt.show()

In [ ]:
# Double check
import matplotlib.cm as cm

# Generate a list of colors using the 'rainbow' colormap
colors = cm.rainbow(np.linspace(0, 1, len(centered_list)))

plt.figure()
for i, value in enumerate(centered_list):
    plt.plot(distances_dict[f'distances_{value}deg'][0], color=colors[i], label=f'{value} deg')
plt.legend()
plt.show()

In [ ]:
for value in centered_list:
    output_obj[f'intensity_{value}deg'].append(np.array(intensity_dict[f'intensity_{value}deg']).T)
    output_obj[f'distances_{value}deg'].append(np.array(distances_dict[f'distances_{value}deg'][0]))

In [ ]:
output_obj.keys()

In [ ]:
# import pickle

# # Export to a pickle file
# with open('./jplot_aia.pkl', 'wb') as pickle_file:
#     pickle.dump(output_obj, pickle_file)

# # Import from a pickle file
# with open('./jplot_aia.pkl', 'rb') as pickle_file:
#     output_obj = pickle.load(pickle_file)

In [ ]:
datenum_arr = [mdates.date2num(pd.Timestamp(str(t))) for t in output_obj['time']]

for value in centered_list:
    
    # conversion from arcsec to solar radius
    dist_rsun = output_obj[f'distances_{value}deg'][0] / output_obj['map_obj'][0].rsun_obs.value
    
    fig = plt.figure(figsize=[8,5])
    
    ax = fig.add_subplot(111)
    ax.imshow(output_obj[f'intensity_{value}deg'][0],
              aspect='auto', origin='lower', cmap='seismic',
                    vmin=-0.5, vmax=3,
            extent=[datenum_arr[0], datenum_arr[-1],
                    dist_rsun[0], dist_rsun[-1]])

    for col in datenum_arr:
        ax.axvline(x=col, color='w', linewidth=0.5, linestyle='--', alpha=0.5)
    
    ax.set_xlabel('Time (UT)')
    ax.set_ylabel(r'Height ($R_\odot$)')
    ax.set_title(f'Slit {value} deg')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_minor_locator(AutoMinorLocator(n=6))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.set_ylim(bottom=1)
#     ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:20:00"),
#                right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:55:00"))
    plt.show()

## Clicking ...

In [ ]:
import matplotlib
matplotlib.use('nbAgg')

# Initialize each key with an empty list
feature_coords_slit = {}
for value in centered_list:
    feature_coords_slit[f'x_{value}deg'] = []
    feature_coords_slit[f'y_{value}deg'] = []


# Define the start and end timestamps
start_timestamp = pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:16:00")
end_timestamp = pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:40:00")

# Generate the list of timestamps with a 1-minute interval
timestamps = pd.date_range(start=start_timestamp, end=end_timestamp, freq='T')
timestamp_list = timestamps.tolist()


# Function to display the figure and handle clicks
def show_figure(angle):
    fig = plt.figure(figsize=[8,5])
    ax = fig.add_subplot(111)
    ax.imshow(output_obj[f'intensity_{angle}deg'][0],
              aspect='auto', origin='lower', cmap='seismic',
              vmin=-0.5, vmax=3,
              extent=[datenum_arr[0], datenum_arr[-1],
                      output_obj[f'distances_{angle}deg'][0][0],
                      output_obj[f'distances_{angle}deg'][0][-1]])

    for col in timestamp_list:
        ax.axvline(x=col, color='w', linewidth=0.5, linestyle='--', alpha=0.5)

    ax.set_xlabel('Time (UT)')
    ax.set_ylabel('Angular distance (arcsec)')
    ax.set_title(f'Slit {angle} deg')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
#     ax.xaxis.set_minor_locator(AutoMinorLocator(n=6))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.set_ylim(bottom=round(output_obj['map_obj'][0].rsun_obs.value))
    ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:16:00"),
                right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:40:00"))
    
    def onclick(event):
        if event.inaxes == ax:
            if event.button == 1:
                x, y = event.xdata, event.ydata
                feature_coords_slit[f'x_{angle}deg'].append(x)
                feature_coords_slit[f'y_{angle}deg'].append(y)
                ax.plot(x, y, 'ko')
                fig.canvas.draw()
            elif event.button == 3:
                fig.canvas.mpl_disconnect(cid)
                plt.close(fig)  # Close the figure
    
    cid = fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show()

In [ ]:
# Iterate over each angle and show the figure
for angle in centered_list:
    show_figure(angle)

In [ ]:
feature_coords_slit

In [ ]:
# Find the maximum length of the arrays in the dictionary
max_length = max(len(coords) for coords in feature_coords_slit.values())

# Pad each array with NaNs to match the maximum length
for key in feature_coords_slit:
    current_length = len(feature_coords_slit[key])
    if current_length < max_length:
        feature_coords_slit[key].extend([np.nan] * (max_length - current_length))

# Convert the dictionary to a DataFrame
df_jplot = pd.DataFrame(feature_coords_slit)

In [ ]:
df_jplot

In [ ]:
# Initialize the dictionary to store grouped x and y values
grouped_coords = {}

# Iterate through the DataFrame columns
for col in df_jplot.columns:
    # Extract the angle from the column name (e.g., 'x_149deg' -> '149deg')
    angle = col.split('_')[1]
    
    # Initialize a sub-dictionary for each angle if it doesn't exist
    if angle not in grouped_coords:
        grouped_coords[angle] = {'x': [], 'y': []}
    
    # Append the values to the respective 'x' or 'y' list
    if col.startswith('x_'):
        grouped_coords[angle]['x'] = df_jplot[col].tolist()
    elif col.startswith('y_'):
        grouped_coords[angle]['y'] = df_jplot[col].tolist()

In [ ]:
grouped_coords['149deg']

## Fitting the points

In [ ]:
%matplotlib inline


for angle in centered_list:
    
    # Extract time and distance arrays from the selected coordinates
    times_num = pd.DataFrame(grouped_coords[f'{angle}deg']).dropna()['x']
    distances = pd.DataFrame(grouped_coords[f'{angle}deg']).dropna()['y']
    
    try:
        # Perform linear regression to fit a line
        slope, intercept, r_value, p_value, std_err = stats.linregress(times_num, distances)

        # get the radius of the solar disk
        sol_rad = const.equatorial_radius.to(u.km)

        # conversion factor from arcsec to km
        conversion_factor = sol_rad/output_obj['map_obj'][0].rsun_obs

        # convert distance from arcsec to km
        distance_km = df_jplot[f'y_{angle}deg'] * conversion_factor

        # calculate the distance difference in km
        distance_diff_km = distance_km.diff()

        # convert time to datetime format
        datetime_obj = [mdates.num2date(t) for t in df_jplot[f'x_{angle}deg'] if not pd.isna(t)]

        # Convert the list of datetime objects to a pandas Series
        datetime_series = pd.Series(datetime_obj)

        # calculate the time difference in seconds
        time_diff_s = datetime_series.diff().dt.total_seconds()

        # calculate the speed in km/s
        df_jplot[f'linear_speed_{angle}deg'] = distance_diff_km / time_diff_s
    except:
        print('\nlinear fit is not possible')
        pass

    # Calculate spline fit
    try:
        spline = UnivariateSpline(times_num, distances, k=1, s=0)  # s=0 for interpolation through all points

        # Generate the spline line
        spline_times = np.linspace(min(times_num), max(times_num), 1000)
        spline_distances = spline(spline_times)

        # Calculate the derivative of spline_distances with respect to spline_times
        spline_velocity = spline.derivative()(spline_times)

        # Convert velocity (in arcsec/day) to speed in km/s
        # 1 arcsec ≈ 733 km on the Sun's surface
        # 1 day = 86400 seconds
        speed_spline = spline_velocity * conversion_factor.value / 86400  # km/s
        df_jplot[f'spline_speed_{angle}deg'] = speed_spline
        df_jplot[f'spline_times_{angle}deg'] = spline_times
        df_jplot[f'spline_dist_{angle}deg'] = spline_distances
    except:
        print('\nspline fit is not possible')
        pass

    # Calculate the polynomial fit
    try:
        polyfit_coeff = np.polyfit(times_num, distances, 2)  # Fit a 2nd order polynomial
        polyfit_line = np.polyval(polyfit_coeff, spline_times)

        # Calculate the derivative of the polynomial fit (velocity)
        polyfit_velocity = np.polyval(np.polyder(polyfit_coeff), spline_times)
        speed_polyfit = polyfit_velocity * conversion_factor.value / 86400  # km/s
        df_jplot[f'polyfit_dist_{angle}deg'] = polyfit_line
        df_jplot[f'polyfit_speed_{angle}deg'] = speed_polyfit
    except:
        print('polynomial fit is not possible')
        pass
    
    
    # Show the plot with the fit lines
    fig = plt.figure(figsize=[8,5])
    ax = fig.add_subplot(111)
    ax.imshow(output_obj[f'intensity_{angle}deg'][0],
              aspect='auto', origin='lower', cmap='seismic',
              vmin=-0.5, vmax=3,
              extent=[datenum_arr[0], datenum_arr[-1],
                      output_obj[f'distances_{angle}deg'][0][0],
                      output_obj[f'distances_{angle}deg'][0][-1]])
    
    # Plot the linear fit line
    if f'linear_speed_{angle}deg' in df_jplot.columns:
        fit_line = slope * np.array(times_num) + intercept

        # Calculate the speed (slope in arcsec/day to speed in km/s)
        # 1 arcsec ≈ 733 km on the Sun's surface
        # 1 day = 86400 seconds
        speed_fit = slope * conversion_factor.value/86400  # km/s
        speed = np.nanmean(df_jplot[f'linear_speed_{angle}deg'])

        # Plot the selected points
        ax.plot(times_num, distances, 'ko', label=f'Mean speed = {speed:.2f} km/s')
        ax.plot(times_num, fit_line, ls='--', color='yellow',
                label=f'Linear fit: {speed_fit:.2f} km/s')

    # Plot the spline fit line
    if f'spline_speed_{angle}deg' in df_jplot.columns:
        spline_times = df_jplot[f'spline_times_{angle}deg']
        spline_distances = df_jplot[f'spline_dist_{angle}deg']
        speed_spline = df_jplot[f'spline_speed_{angle}deg']

        ax.plot(spline_times, spline_distances, ls='--', color='tab:red',
                label=f'Spline fit: {np.nanmean(speed_spline):.2f} km/s')

    # Plot the polynomial fit line
    if f'polyfit_speed_{angle}deg' in df_jplot.columns:
        spline_times = df_jplot[f'spline_times_{angle}deg']
        polyfit_distances = df_jplot[f'polyfit_dist_{angle}deg']
        speed_polyfit = df_jplot[f'polyfit_speed_{angle}deg']

        ax.plot(spline_times, polyfit_distances, ls='-', color='tab:blue',
                   label=f'2nd-order Polynomial fit: {np.nanmean(speed_polyfit):.2f} km/s')

    ax.legend()
    ax.set_xlabel('Time (UT)')
    ax.set_ylabel('Angular distance (arcsec)')
    ax.set_title(f'Slit {angle} deg')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_minor_locator(AutoMinorLocator(n=6))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.set_ylim(bottom=round(output_obj['map_obj'][0].rsun_obs.value))
    ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:20:00"),
                right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:55:00"))

    fig.tight_layout()
    plt.show()

In [ ]:
df_jplot

In [ ]:
# export data points as tables
df_jplot.to_csv('../dias_work/jplot_aia_CME_no2.csv')

---

## Working with single channel

In [ ]:
# Store lists in a dictionary
lists_dict = {
    171: m_seq_runratio_171A,
    193: m_seq_runratio_193A,
    211: m_seq_runratio_211A
}

def get_list_copy(number):
    list_to_copy = lists_dict.get(number)
    if list_to_copy is not None:
        return deepcopy(list_to_copy)
    else:
        return None

In [ ]:
channel = 171
final_maps_list = get_list_copy(channel)
cme_number = 2

In [ ]:
m = deepcopy(final_maps_list[6])

slits = False
norm = ImageNormalize(vmin=0.7, vmax=1.5, stretch=SqrtStretch())

smoothed_data = gaussian_filter(m.data, sigma=1)
smoothed_map = sunpy.map.Map(smoothed_data, m.meta)

fig = plt.figure(figsize=[10,10])
ax = fig.add_subplot(111, projection=m)
img = smoothed_map.plot(axes=ax, cmap='Greys_r', norm=norm)
fig.colorbar(img, shrink=0.5, pad=0.02)

if slits:
    centered_list = generate_centered_list(160, 2, 3)
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1450, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 50, end_point_pixel.y.value + 50, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')

plt.show()

In [ ]:
%matplotlib inline

# Create a dictionary to hold the lists of lists
intensity_dict = {}
distances_dict = {}
output_obj = {}

# Initialize each key with an empty list
for value in centered_list:
    intensity_dict[f'intensity_{value}deg'] = []
    distances_dict[f'distances_{value}deg'] = []
    
    output_obj[f'intensity_{value}deg'] = []
    output_obj[f'distances_{value}deg'] = []

output_obj['time'] = []
output_obj['map_obj'] = []
output_obj['instrument'] = []

centered_list = generate_centered_list(160, 2, 3)

for i, m in enumerate(final_maps_list):
    print(f'Working on map {i} ..')
    
    norm = ImageNormalize(vmin=0.7, vmax=1.5, stretch=SqrtStretch())
    smoothed_data = gaussian_filter(m.data, sigma=1)
    smoothed_map = sunpy.map.Map(smoothed_data, m.meta)
    
    fig = plt.figure(figsize=[7,7])
    ax = fig.add_subplot(111, projection=m)
    smoothed_map.plot(axes=ax, cmap='Greys_r', norm=norm)
    
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1450, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 50, end_point_pixel.y.value + 50, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
        
        # obtain the coordinates of the map pixels that intersect that path
        intensity_coords_slit = sunpy.map.pixelate_coord_path(m, line)
        
        # Create mask to identify valid coordinates
        valid_mask = sunpy.map.contains_coordinate(m, intensity_coords_slit)
        
        # Apply the mask to filter valid coordinates
        valid_coords = intensity_coords_slit[valid_mask]
        
        # Pass those coordinates to extract the values for those map pixels
        intensity_slit = sunpy.map.sample_at_coords(m, valid_coords)
        
        # Calculate the angular separation between the first point and every other coordinate we extracted
        angular_separation_slit = valid_coords.separation(valid_coords[0]).to(u.arcsec)
        
        # Append the values to the lists
        intensity_dict[f'intensity_{value}deg'].append(list(intensity_slit.value))
        distances_dict[f'distances_{value}deg'].append(list(angular_separation_slit.value))
    
    output_obj['time'].append(m.date.iso)
    output_obj['map_obj'].append(m)
    output_obj['instrument'].append(m.meta['instrume'])
    
    plt.show()

In [ ]:
import matplotlib
matplotlib.use('nbAgg')

In [ ]:
for value in centered_list:
    output_obj[f'intensity_{value}deg'].append(np.array(intensity_dict[f'intensity_{value}deg']).T)
    output_obj[f'distances_{value}deg'].append(np.array(distances_dict[f'distances_{value}deg'][0]))

# Initialize each key with an empty list
feature_coords_slit = {}
for value in centered_list:
    feature_coords_slit[f'x_{value}deg'] = []
    feature_coords_slit[f'y_{value}deg'] = []


# Define the start and end timestamps
start_timestamp = pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:16:00")
end_timestamp = pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:40:00")

# Generate the list of timestamps with a 1-minute interval
timestamps = pd.date_range(start=start_timestamp, end=end_timestamp, freq='T')
timestamp_list = timestamps.tolist()


# Function to display the figure and handle clicks
def show_figure(angle):
    fig = plt.figure(figsize=[8,5])
    ax = fig.add_subplot(111)
    ax.imshow(output_obj[f'intensity_{angle}deg'][0],
              aspect='auto', origin='lower', cmap='Greys',
              vmin=0.5, vmax=1.1,
              extent=[datenum_arr[0], datenum_arr[-1],
                      output_obj[f'distances_{angle}deg'][0][0],
                      output_obj[f'distances_{angle}deg'][0][-1]])

    for col in timestamp_list:
        ax.axvline(x=col, color='w', linewidth=0.5, linestyle='--', alpha=0.5)

    ax.set_xlabel('Time (UT)')
    ax.set_ylabel('Angular distance (arcsec)')
    ax.set_title(f'Slit {angle} deg')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.set_ylim(bottom=round(output_obj['map_obj'][0].rsun_obs.value))
    ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:16:00"),
                right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:40:00"))
    
    def onclick(event):
        if event.inaxes == ax:
            if event.button == 1:
                x, y = event.xdata, event.ydata
                feature_coords_slit[f'x_{angle}deg'].append(x)
                feature_coords_slit[f'y_{angle}deg'].append(y)
                ax.plot(x, y, 'rx', markersize=7)
                fig.canvas.draw()
            elif event.button == 3:
                fig.canvas.mpl_disconnect(cid)
                plt.close(fig)  # Close the interaction with the figure
    
    cid = fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show()

In [ ]:
# Iterate over each angle and show the figure
for angle in centered_list:
    show_figure(angle)

In [ ]:
%matplotlib inline


# Find the maximum length of the arrays in the dictionary
max_length = max(len(coords) for coords in feature_coords_slit.values())

# Pad each array with NaNs to match the maximum length
for key in feature_coords_slit:
    current_length = len(feature_coords_slit[key])
    if current_length < max_length:
        feature_coords_slit[key].extend([np.nan] * (max_length - current_length))

# Convert the dictionary to a DataFrame
df_jplot = pd.DataFrame(feature_coords_slit)

# Initialize the dictionary to store grouped x and y values
grouped_coords = {}

# Iterate through the DataFrame columns
for col in df_jplot.columns:
    # Extract the angle from the column name (e.g., 'x_149deg' -> '149deg')
    angle = col.split('_')[1]
    
    # Initialize a sub-dictionary for each angle if it doesn't exist
    if angle not in grouped_coords:
        grouped_coords[angle] = {'x': [], 'y': []}
    
    # Append the values to the respective 'x' or 'y' list
    if col.startswith('x_'):
        grouped_coords[angle]['x'] = df_jplot[col].tolist()
    elif col.startswith('y_'):
        grouped_coords[angle]['y'] = df_jplot[col].tolist()




for angle in centered_list:
    
    # Extract time and distance arrays from the selected coordinates
    times_num = pd.DataFrame(grouped_coords[f'{angle}deg']).dropna()['x']
    distances = pd.DataFrame(grouped_coords[f'{angle}deg']).dropna()['y']
    
    try:
        # Perform linear regression to fit a line
        slope, intercept, r_value, p_value, std_err = stats.linregress(times_num, distances)

        # get the radius of the solar disk
        sol_rad = const.equatorial_radius.to(u.km)

        # conversion factor from arcsec to km
        conversion_factor = sol_rad/output_obj['map_obj'][0].rsun_obs

        # convert distance from arcsec to km
        distance_km = df_jplot[f'y_{angle}deg'] * conversion_factor

        # calculate the distance difference in km
        distance_diff_km = distance_km.diff()

        # convert time to datetime format
        datetime_obj = [mdates.num2date(t) for t in df_jplot[f'x_{angle}deg'] if not pd.isna(t)]

        # Convert the list of datetime objects to a pandas Series
        datetime_series = pd.Series(datetime_obj)

        # calculate the time difference in seconds
        time_diff_s = datetime_series.diff().dt.total_seconds()

        # calculate the speed in km/s
        df_jplot[f'linear_speed_{angle}deg'] = distance_diff_km / time_diff_s
    except:
        print('\nlinear fit is not possible')
        pass

    # Calculate spline fit
    try:
        spline = UnivariateSpline(times_num, distances, k=1, s=0)  # s=0 for interpolation through all points

        # Generate the spline line
        spline_times = np.linspace(min(times_num), max(times_num), 1000)
        spline_distances = spline(spline_times)

        # Calculate the derivative of spline_distances with respect to spline_times
        spline_velocity = spline.derivative()(spline_times)

        # Convert velocity (in arcsec/day) to speed in km/s
        # 1 arcsec ≈ 733 km on the Sun's surface
        # 1 day = 86400 seconds
        speed_spline = spline_velocity * conversion_factor.value / 86400  # km/s
        df_jplot[f'spline_speed_{angle}deg'] = speed_spline
        df_jplot[f'spline_times_{angle}deg'] = spline_times
        df_jplot[f'spline_dist_{angle}deg'] = spline_distances
    except:
        print('\nspline fit is not possible')
        pass

    # Calculate the polynomial fit
    try:
        polyfit_coeff = np.polyfit(times_num, distances, 2)  # Fit a 2nd order polynomial
        polyfit_line = np.polyval(polyfit_coeff, spline_times)

        # Calculate the derivative of the polynomial fit (velocity)
        polyfit_velocity = np.polyval(np.polyder(polyfit_coeff), spline_times)
        speed_polyfit = polyfit_velocity * conversion_factor.value / 86400  # km/s
        df_jplot[f'polyfit_dist_{angle}deg'] = polyfit_line
        df_jplot[f'polyfit_speed_{angle}deg'] = speed_polyfit
    except:
        print('polynomial fit is not possible')
        pass
    
    
    # Show the plot with the fit lines
    fig = plt.figure(figsize=[8,5])
    ax = fig.add_subplot(111)
    ax.imshow(output_obj[f'intensity_{angle}deg'][0],
              aspect='auto', origin='lower', cmap='Greys',
              vmin=0.5, vmax=1.1,
              extent=[datenum_arr[0], datenum_arr[-1],
                      output_obj[f'distances_{angle}deg'][0][0],
                      output_obj[f'distances_{angle}deg'][0][-1]])
    
    # Plot the linear fit line
    if f'linear_speed_{angle}deg' in df_jplot.columns:
        fit_line = slope * np.array(times_num) + intercept

        # Calculate the speed (slope in arcsec/day to speed in km/s)
        # 1 arcsec ≈ 733 km on the Sun's surface
        # 1 day = 86400 seconds
        speed_fit = slope * conversion_factor.value/86400  # km/s
        speed = np.nanmean(df_jplot[f'linear_speed_{angle}deg'])

        # Plot the selected points
        ax.plot(times_num, distances, 'rx', markersize=7, label=f'Mean speed = {speed:.2f} km/s')
        ax.plot(times_num, fit_line, ls='--', color='yellow',
                label=f'Linear fit: {speed_fit:.2f} km/s')

    # Plot the spline fit line
    if f'spline_speed_{angle}deg' in df_jplot.columns:
        spline_times = df_jplot[f'spline_times_{angle}deg']
        spline_distances = df_jplot[f'spline_dist_{angle}deg']
        speed_spline = df_jplot[f'spline_speed_{angle}deg']

        ax.plot(spline_times, spline_distances, ls='--', color='tab:red',
                label=f'Spline fit: {np.nanmean(speed_spline):.2f} km/s')

    # Plot the polynomial fit line
    if f'polyfit_speed_{angle}deg' in df_jplot.columns:
        spline_times = df_jplot[f'spline_times_{angle}deg']
        polyfit_distances = df_jplot[f'polyfit_dist_{angle}deg']
        speed_polyfit = df_jplot[f'polyfit_speed_{angle}deg']

        ax.plot(spline_times, polyfit_distances, ls='-', color='tab:blue',
                   label=f'2nd-order Polynomial fit: {np.nanmean(speed_polyfit):.2f} km/s')

    ax.legend()
    ax.set_xlabel('Time (UT)')
    ax.set_ylabel('Angular distance (arcsec)')
    ax.set_title(f'Slit {angle} deg')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_minor_locator(AutoMinorLocator(n=6))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.set_ylim(bottom=round(output_obj['map_obj'][0].rsun_obs.value))
    ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:16:00"),
                right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:40:00"))

    fig.tight_layout()
    plt.show()


# export data points as tables
df_jplot.to_csv(f'../dias_work/jplot_aia_{channel}_CME_No{cme_number}.csv')